# ScanNet Scene Downloader
Downloads `.sens` files for selected ScanNet scenes directly to Google Drive, extracts them into the `color/`, `depth/`, `pose/`, `intrinsic/` format expected by the training pipeline, then deletes the raw `.sens` to save space.

**Expected Drive layout after running:**
```
MyDrive/final_proj/scannet/scans/
  scene0001_00/exported/{color, depth, pose, intrinsic}/
  scene0002_00/exported/{color, depth, pose, intrinsic}/
  ...
```

> `scene0000_00` is already downloaded locally — skip it here.

In [9]:
# ── Cell 1: Mount Drive ───────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/final_proj/scannet/scans'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive mounted. Scans will be saved to:', DRIVE_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted. Scans will be saved to: /content/drive/MyDrive/final_proj/scannet/scans


In [ ]:
                                                                                     
  ! cd /Users/siddharthraj/classes/cv/cv_final/scannet-data-extract &&                
  /Users/siddharthraj/classes/cv/cv_final/myenv/bin/python extract_scannet.py --scenes
   scene0012_00

In [10]:
# ── Cell 2: Install deps & copy scannet.py from Drive ────────────────────────
!pip install numpy opencv-python-headless tqdm --quiet

import sys, shutil

# Copy scannet.py from Drive so we can import its functions directly
SCANNET_PY_SRC = '/content/drive/MyDrive/final_proj/scannet.py'
assert os.path.exists(SCANNET_PY_SRC), \
    'Upload scannet.py to MyDrive/final_proj/scannet.py first!'

shutil.copy(SCANNET_PY_SRC, '/content/scannet.py')
sys.path.insert(0, '/content')

# Import download_scan directly — bypasses all the interactive input() prompts
# that only live in main()
from scannet import download_scan

print('scannet.py imported. download_scan() ready.')

scannet.py imported. download_scan() ready.


In [11]:
# ── Cell 3: Configure scenes to download ─────────────────────────────────────
# scene0000_00 is already downloaded locally — start from scene0001_00.
# Adjust this list to download more or fewer scenes.
# Each .sens is ~1-3 GB; extracted exported/ is ~500 MB per scene.
# 10 scenes ≈ 5 GB extracted on Drive.

SCENES = [
    'scene0001_00',
    'scene0002_00',
    'scene0003_00',
    'scene0004_00',
    'scene0005_00',
    'scene0006_00',
    'scene0007_00',
    'scene0008_00',
    'scene0009_00',
    'scene0010_00',
]

print(f'Will download and extract {len(SCENES)} scenes:')
for s in SCENES:
    exported = os.path.join(DRIVE_ROOT, s, 'exported')
    status = '✓ already done' if os.path.exists(exported) else '  pending'
    print(f'  {status}  {s}')

Will download and extract 10 scenes:
    pending  scene0001_00
    pending  scene0002_00
    pending  scene0003_00
    pending  scene0004_00
    pending  scene0005_00
    pending  scene0006_00
    pending  scene0007_00
    pending  scene0008_00
    pending  scene0009_00
    pending  scene0010_00


In [22]:
# ── Cell 4: Download + extract each scene ────────────────────────────────────
import os, struct, zlib, ssl, sys, shutil
import cv2
import numpy as np
from tqdm import tqdm as tqdm_bar

ssl._create_default_https_context = ssl._create_unverified_context

SCANNET_PY_SRC = '/content/drive/MyDrive/final_proj/scannet.py'
assert os.path.exists(SCANNET_PY_SRC)
shutil.copy(SCANNET_PY_SRC, '/content/scannet.py')
sys.path.insert(0, '/content')
from scannet import download_scan
print('download_scan() ready.')

# ── Raw header dump (run once to calibrate) ───────────────────────────────────
def dump_header(filename):
    with open(filename, 'rb') as f:
        v = struct.unpack('I', f.read(4))[0];   print(f'version={v}')
        sl = struct.unpack('Q', f.read(8))[0];  print(f'strlen(Q)={sl}')
        sn = f.read(sl);                        print(f'scene_name={sn}')
        pos = f.tell()
        print(f'\n-- raw bytes from offset {pos}, shown as overlapping windows --')
        chunk = f.read(200)
        for off in range(0, min(160, len(chunk)-8), 4):
            b4  = chunk[off:off+4]
            b8  = chunk[off:off+8]
            i4  = struct.unpack('i', b4)[0]
            I4  = struct.unpack('I', b4)[0]
            q8  = struct.unpack('Q', b8)[0] if len(b8)==8 else -1
            print(f'  +{off:3d}  i={i4:12d}  I={I4:12d}  Q={q8:20d}  hex4={b4.hex()}')

COMPRESSION_TYPE_COLOR = {-1:'unknown', 0:'raw', 1:'png', 2:'jpeg'}
COMPRESSION_TYPE_DEPTH = {-1:'unknown', 0:'raw_ushort', 1:'zlib_ushort', 2:'occi_ushort'}

class RGBDFrame:
    def load(self, f, num_bytes_intrinsics):
        self.camera_intrinsics = np.frombuffer(f.read(num_bytes_intrinsics), dtype=np.float32).copy()
        self.camera_to_world   = np.frombuffer(f.read(64), dtype=np.float32).reshape(4, 4).copy()
        self.timestamp_color   = struct.unpack('Q', f.read(8))[0]
        self.timestamp_depth   = struct.unpack('Q', f.read(8))[0]
        self.color_size_bytes  = struct.unpack('Q', f.read(8))[0]
        self.depth_size_bytes  = struct.unpack('Q', f.read(8))[0]
        self.color_data        = f.read(self.color_size_bytes)
        self.depth_data        = f.read(self.depth_size_bytes)

class SensorData:
    def __init__(self, filename):
        with open(filename, 'rb') as f:
            version = struct.unpack('I', f.read(4))[0]
            assert version == 4, f'Unsupported .sens version {version}'
            strlen = struct.unpack('Q', f.read(8))[0]
            f.read(strlen)
            print(f'  after scene_name, file offset = {f.tell()}')

            self.num_bytes_intrinsics = struct.unpack('Q', f.read(8))[0]
            print(f'  num_bytes_intrinsics = {self.num_bytes_intrinsics}')

            self.num_frames = struct.unpack('Q', f.read(8))[0]
            print(f'  num_frames = {self.num_frames}')

            _nbpf = struct.unpack('Q', f.read(8))[0]
            print(f'  num_bytes_per_frame = {_nbpf}')

            raw4 = f.read(4)
            color_int = struct.unpack('i', raw4)[0]
            print(f'  color_compression raw={raw4.hex()}  int={color_int}')
            self.color_compression = COMPRESSION_TYPE_COLOR[color_int]

            raw4 = f.read(4)
            depth_int = struct.unpack('i', raw4)[0]
            print(f'  depth_compression raw={raw4.hex()}  int={depth_int}')
            self.depth_compression = COMPRESSION_TYPE_DEPTH[depth_int]

            self.color_width  = struct.unpack('I', f.read(4))[0]
            self.color_height = struct.unpack('I', f.read(4))[0]
            self.depth_width  = struct.unpack('I', f.read(4))[0]
            self.depth_height = struct.unpack('I', f.read(4))[0]
            self.depth_shift  = struct.unpack('f', f.read(4))[0]
            print(f'  color={self.color_width}x{self.color_height}  depth={self.depth_width}x{self.depth_height}  shift={self.depth_shift}')

            self.frames = []
            print(f'  Loading {self.num_frames} frames...')
            for i in range(self.num_frames):
                frame = RGBDFrame()
                frame.load(f, self.num_bytes_intrinsics)
                self.frames.append(frame)

    def export(self, out_dir):
        color_dir     = os.path.join(out_dir, 'color')
        depth_dir     = os.path.join(out_dir, 'depth')
        pose_dir      = os.path.join(out_dir, 'pose')
        intrinsic_dir = os.path.join(out_dir, 'intrinsic')
        for d in [color_dir, depth_dir, pose_dir, intrinsic_dir]:
            os.makedirs(d, exist_ok=True)
        K = self.frames[0].camera_intrinsics.reshape(4, 4)
        np.savetxt(os.path.join(intrinsic_dir, 'intrinsic_depth.txt'), K)
        np.savetxt(os.path.join(intrinsic_dir, 'intrinsic_color.txt'), K)
        for i, frame in enumerate(tqdm_bar(self.frames, desc='  Extracting')):
            with open(os.path.join(color_dir, f'{i}.jpg'), 'wb') as cf:
                cf.write(frame.color_data)
            raw   = zlib.decompress(frame.depth_data) if self.depth_compression == 'zlib_ushort' else frame.depth_data
            depth = np.frombuffer(raw, dtype=np.uint16).reshape(self.depth_height, self.depth_width)
            cv2.imwrite(os.path.join(depth_dir, f'{i}.png'), depth)
            np.savetxt(os.path.join(pose_dir, f'{i}.txt'), frame.camera_to_world)

# ── Main loop ─────────────────────────────────────────────────────────────────
MIN_SENS_BYTES = 50 * 1024 * 1024
SCRATCH = '/content/scratch'
os.makedirs(SCRATCH, exist_ok=True)

for scene_id in SCENES:
    exported_dir = os.path.join(DRIVE_ROOT, scene_id, 'exported')
    color_dir    = os.path.join(exported_dir, 'color')

    if os.path.exists(color_dir) and len(os.listdir(color_dir)) > 100:
        print(f'[SKIP] {scene_id} — already extracted ({len(os.listdir(color_dir))} frames)')
        continue

    print(f'\n── {scene_id} ──')
    sens_path = os.path.join(SCRATCH, f'{scene_id}.sens')

    if os.path.exists(sens_path):
        size = os.path.getsize(sens_path)
        if size < MIN_SENS_BYTES:
            print(f'  Deleting corrupt .sens ({size/1024:.1f} KB)')
            os.remove(sens_path)
        else:
            print(f'  .sens on scratch ({size/1024/1024:.0f} MB)')

    if not os.path.exists(sens_path):
        download_scan(scene_id, SCRATCH, file_types=['.sens'], use_v1_sens=True, skip_existing=False)

    # Dump header to diagnose any format issues
    print(f'\n  === Header dump for {scene_id} ===')
    dump_header(sens_path)
    print()

    sd = SensorData(sens_path)
    sd.export(exported_dir)
    os.remove(sens_path)
    print(f'  Done — {len(os.listdir(color_dir))} frames written to Drive.')

print('\n=== All scenes processed ===')

download_scan() ready.

── scene0001_00 ──
  .sens on scratch (542 MB)

  === Header dump for scene0001_00 ===
version=4
strlen(Q)=15
scene_name=b'StructureSensor'

-- raw bytes from offset 27, shown as overlapping windows --
  +  0  i=  1150436868  I=  1150436868  Q=          1150436868  hex4=04469244
  +  4  i=           0  I=           0  Q= 4909468951601217536  hex4=00000000
  +  8  i=  1143074816  I=  1143074816  Q=          1143074816  hex4=00f02144
  + 12  i=           0  I=           0  Q=                   0  hex4=00000000
  + 16  i=           0  I=           0  Q= 4941088724172668928  hex4=00000000
  + 20  i=  1150436868  I=  1150436868  Q= 4895940561683498500  hex4=04469244
  + 24  i=  1139924992  I=  1139924992  Q=          1139924992  hex4=00e0f143
  + 28  i=           0  I=           0  Q=                   0  hex4=00000000
  + 32  i=           0  I=           0  Q=                   0  hex4=00000000
  + 36  i=           0  I=           0  Q= 4575657221408423936  hex4=000

KeyError: 1139924992

In [ ]:
# ── Debug cell: print raw header bytes to determine exact .sens format ────────
import struct, os

sens_path = '/content/scratch/scene0001_00.sens'
assert os.path.exists(sens_path) and os.path.getsize(sens_path) > 1e6, \
    'Run Cell 4 first (let it fail after downloading) or check path'

with open(sens_path, 'rb') as f:
    version = struct.unpack('I', f.read(4))[0]
    print(f'version          = {version}')

    strlen = struct.unpack('Q', f.read(8))[0]
    print(f'strlen (Q,8B)    = {strlen}')
    scene_name = f.read(strlen)
    print(f'scene_name       = {scene_name}')

    # Print next 10 fields as both Q (8B) and I (4B) so we can spot the right type
    print('\n--- Next 128 bytes read as Q (8-byte uint64) ---')
    saved_pos = f.tell()
    for i in range(16):
        raw = f.read(8)
        if len(raw) < 8: break
        val_Q = struct.unpack('Q', raw)[0]
        val_i = struct.unpack('i', raw[:4])[0]
        val_I = struct.unpack('I', raw[:4])[0]
        print(f'  offset={saved_pos + i*8:4d}  Q={val_Q:20d}  i(4B)={val_i:12d}  I(4B)={val_I:12d}  hex={raw.hex()}')

In [ ]:
# ── Cell 5: Verify structure ──────────────────────────────────────────────────
print('Verification:\n')
print(f'{"Scene":<20} {"Color frames":>14} {"Depth frames":>14} {"Poses":>8}')
print('-' * 60)
for scene_id in SCENES:
    exp = os.path.join(DRIVE_ROOT, scene_id, 'exported')
    if not os.path.exists(exp):
        print(f'{scene_id:<20} MISSING')
        continue
    n_color = len(os.listdir(os.path.join(exp, 'color'))) if os.path.exists(os.path.join(exp, 'color')) else 0
    n_depth = len(os.listdir(os.path.join(exp, 'depth'))) if os.path.exists(os.path.join(exp, 'depth')) else 0
    n_pose  = len(os.listdir(os.path.join(exp, 'pose')))  if os.path.exists(os.path.join(exp, 'pose'))  else 0
    ok = '✓' if n_color == n_depth == n_pose and n_color > 0 else '✗'
    print(f'{ok} {scene_id:<18} {n_color:>14} {n_depth:>14} {n_pose:>8}')